# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shouban7866/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. Two Paper Findings + Methodology Audit

Finding 1: The Content Lifecycle & "Decay Cliff"

Where the Label Comes From:
The paper classifies pages into age buckets (0–7 days up to 365+ days) and tracks their performance using FlyRank Health Score (a 0–100 composite metric: Impressions 30pts + Position 30pts + CTR 20pts + Scroll Depth 20pts). It claims content peaks at 61–90 days (Health: 33.1) and hits a "decay cliff" at 271–365 days where performance drops sharply (Health: 14).

Does the Validation Design Carry the Claim?

Partially / Weakly. While the aggregate snapshot reveals a clear observational pattern, relying on the FlyRank Health Score as the primary outcome metric introduces a structural confounder. Because Health Score relies heavily on raw impressions and position, seasonal shifts or portfolio-wide indexing issues can artificially drag down score distributions for older cohorts. Furthermore, using cross-sectional snapshots of different pages across age tiers—rather than a longitudinal cohort tracking the same pages over time—means survivor bias and varying content quality across cohorts could distort the observed decay rate.  
Finding 2: AI Referrals vs. Google Search Rankings (Finding #6)

Where the Label Comes From:

The label comes from GA4 known AI referral traffic tracking (sessions from ChatGPT, Gemini, Perplexity, Copilot, Claude, Meta AI) matched against Google Search Console (GSC) performance data across $N = 341,701$ pages. Pages are segmented into high_ai, some_ai, and no_ai buckets.

Does the Validation Design Carry the Claim?

Yes, with minor caveats. The data shows that the high_ai group averages ~9x more impressions (24.9K vs. 2.7K) despite having a weaker average Google position (19.8 vs. 14.2) compared to the no_ai group. This strongly validates the core claim that AI referral visibility operates as a distinct layer rather than a direct mirror of Google Search rank. The caveat is sample scale: AI traffic accounts for only 1.06% of total portfolio sessions (17,344 out of 1.6M), meaning while the structural behavior is real, its portfolio-level business impact remains modest.  

Methodological Summary & Recommendations+---------------------------------------------------------------------------------------------------+
| Finding                   | Core Metric Used         | Main Design Flaw / Bias                    |
+---------------------------+--------------------------+--------------------------------------------+
| 1. Content Lifecycle      | Composite Health Score   | Cross-sectional cohort bias; reliance on   |
|                            | (0–100)                  | internal composite metric over raw search  |
+---------------------------+--------------------------+--------------------------------------------+
| 2. AI Visibility Layer    | GA4 Referrals vs. GSC    | Low overall base rate (1.06% of total      |
|                            | Impressions & Position   | traffic), though directionally sound      |
+---------------------------+--------------------------+--------------------------------------------+

Constructive Recommendation:

To strengthen future claims, prioritize longitudinal cohort analysis (tracking the same cohort over 12 months) and report raw search performance metrics (clicks, impressions, position) independently of internal composite scoring.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The model was evaluated using a grouped 80/20 train-test split based on client_id. This prevents content from the same client appearing in both the training and testing sets, making the evaluation more honest. A Random Forest Classifier was compared with a baseline classifier using the same accuracy metric.

In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Target
threshold = df["engagement_rate"].quantile(0.75)
df["target"] = (df["engagement_rate"] >= threshold).astype(int)

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "avg_position",
    "scroll_rate"
]

X = df[features].fillna(0)
y = df["target"]

# Honest grouped split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
model_acc = accuracy_score(y_test, model.predict(X_test))

print("Baseline Accuracy:", baseline_acc)
print("Random Forest Accuracy:", model_acc)

Baseline Accuracy: 0.7833847152360863
Random Forest Accuracy: 0.8382281356482233


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set was reviewed for possible data leakage. Identifier fields (content_id and client_id) were excluded from the model features. The target was created from engagement_rate, but engagement_rate was not included in the feature vector. Only information available before prediction was used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Random Forest model achieved higher accuracy than the baseline on the available dataset under the grouped train-test split. This is an observed result on the current data and should be used for decision support rather than as a guarantee of future content performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.